# ESE 6800 · Week 1: OmniJev-0.8B on Colab
**Goal of Week 1:** get OmniJev-0.8B running, reproduce its released accuracy, measure calibration, and find out whether the
vision encoder, the shared prefix or the candidate branches dominate latency. That profile decides where pruning can buy speed.

What this notebook does, in order:

| § | Step | Output (saved to Drive) |
|---|---|---|
| 0 | Config, GPU check, Drive | `env.json` |
| 1 | Install pinned stack (FastJEV's exact versions) | – |
| 2 | Download pinned weights, verify SHA-256 | weights cached on Drive |
| 3 | Smoke test: one image, choice + noul | – |
| 4 | Reproduce AI2D + POPE (dense), accuracy / ECE / Brier | `results/*.jsonl`, `summary.json` |
| 5 | Same eval with FastJEV 6-layer pruning (baseline sanity) | same |
| 6 | Latency breakdown: vision vs prefix vs branches vs head | `profile.json` |
| 7 | 2:4 sparse go / no-go micro-benchmark at real shapes | `sparse_bench.json` |
| 8 | LoRA merge equivalence check | printed max diff |

**Runtime:** pick **L4 GPU** (Runtime → Change runtime type). T4 has no 2:4 sparse tensor cores, so §7 would be meaningless there.
Only compare timings made on the same GPU type in the same session.

Inference goes through FastJEV's `original` mode, which is the released OmniJev branch path, so our numbers use the
same pinned checkpoints as the FastJEV paper (0.8B dense: AI2D 62.27%, POPE 84.76% on an L20).

In [1]:
# 0 Config: edit these
DRIVE_ROOT = "/content/drive/MyDrive/ese6800_omnijev"   # shared folder for weights + results
RUN_NAME   = "week1"
N_EVAL     = 300      # Reduced from None to 300 to avoid Colab timeout limits during evaluation.
SEED       = 0

import os, json, subprocess, sys, platform, time
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(f"{DRIVE_ROOT}/results/{RUN_NAME}", exist_ok=True)
OUT = f"{DRIVE_ROOT}/results/{RUN_NAME}"

gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version,compute_cap",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
print("GPU:", gpu, "| Python", platform.python_version())
assert gpu, "No GPU: Runtime -> Change runtime type -> L4"
if "T4" in gpu:
    print("WARNING: T4 has no 2:4 sparse tensor cores. Fine for accuracy work, not for speed claims.")
json.dump({"gpu": gpu, "python": platform.python_version(), "started": time.ctime()},
          open(f"{OUT}/env.json", "w"), indent=1)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
GPU: NVIDIA L4, 23034 MiB, 580.82.07, 8.9 | Python 3.13.16


## §1 Install the pinned stack
FastJEV pins `torch 2.7.1 / transformers 5.17.0 / peft 0.21.0 / fla-core 0.5.2`; its code depends on these exact Qwen3.5
interfaces. **Do not downgrade transformers to 4.x**: Qwen3.5 (`transformers.models.qwen3_5`) only exists in 5.x, so OmniJev
cannot load on 4.x even if the imports succeed.

**Python 3.13 note.** FastJEV also pins `numpy==1.26.4`, which has no Python 3.13 wheel. A plain `pip install -e FastJEV`
therefore fails part-way, leaving Colab's preinstalled transformers next to a mismatched peft (the
`cannot import name 'resolve_revision'` error). It would also break Colab's scipy / scikit-learn, which are built against numpy 2.
So we install the pins ourselves with numpy 2.x, then install FastJEV with `--no-deps`.

Installing takes ~3–5 min. **Restart the runtime once afterwards** (Runtime → Restart session), then run §0 again and the sanity check.

In [2]:
%cd /content
!test -d FastJEV || git clone -q https://github.com/JieMaMagic/FastJEV
!test -d OmniJev || git clone -q https://github.com/tinnel123666888/OmniJev
!pip -q uninstall -y torchaudio          # preinstalled build is tied to Colab's torch; we don't need it
!pip -q install --index-url https://download.pytorch.org/whl/cu126 torch==2.7.1 torchvision==0.22.1 triton==3.3.1
# one resolver run for everything else, so transformers / peft / hub / datasets agree with each other
!pip -q install "transformers==5.17.0" "peft==0.21.0" "accelerate==1.15.0" "safetensors==0.8.0" "fla-core==0.5.2" \
    "numpy>=2.1,<2.4" "pillow>=11" datasets "huggingface_hub[hf_xet]"
!pip -q install --no-deps -e /content/FastJEV
!pip check || true                        # prints any remaining conflicts; FastJEV's numpy pin is expected to show

/content
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for fastjev-inference (pyproject.toml) ... done
ipython 7.34.0 requires jedi, which is not installed.
fastjev-inference 0.1.0 has requirement numpy==1.26.4, but you have numpy 2.3.5.
fastjev-inference 0.1.0 has requirement pillow==12.3.0, but you have pillow 11.3.0.


In [7]:
# Install a version of torchao compatible with peft's requirements
!pip -q install "torchao>=0.17.0" --no-deps

In [4]:
# After restart: sanity-check the stack. Every line must print OK.
import importlib, sys, platform
print("Python", platform.python_version())
want = {"torch": "2.7.1", "transformers": "5.17.0", "peft": "0.21.0", "accelerate": "1.15.0"}
for mod, v in want.items():
    got = importlib.import_module(mod).__version__.split("+")[0]
    print(("OK  " if got == v else "BAD ") + f"{mod} {got} (want {v})")
for mod in ("numpy", "scipy", "sklearn", "datasets", "huggingface_hub"):
    m = importlib.import_module(mod); print("OK  ", mod, m.__version__)
import torch
print("OK   CUDA", torch.version.cuda, "| GPU", torch.cuda.get_device_name(), "| cap", torch.cuda.get_device_capability())
import transformers.models.qwen3_5.modeling_qwen3_5        # fails on transformers 4.x: OmniJev cannot load there
print("OK   transformers has Qwen3.5")
from peft import PeftModel; print("OK   peft imports against this transformers")
from fla.ops import chunk_gated_delta_rule; print("OK   fla-core kernels")
sys.path.insert(0, "/content/FastJEV"); import fastjev; from mso.infer import MSO1; print("OK   FastJEV + OmniJev code")

Python 3.13.16
OK  torch 2.7.1 (want 2.7.1)
OK  transformers 5.17.0 (want 5.17.0)
OK  peft 0.21.0 (want 0.21.0)
OK  accelerate 1.15.0 (want 1.15.0)
OK   numpy 2.4.6
OK   scipy 1.16.3
OK   sklearn 1.6.1
OK   datasets 4.8.5
OK   huggingface_hub 1.33.0
OK   CUDA 12.6 | GPU NVIDIA L4 | cap (8, 9)
OK   transformers has Qwen3.5
OK   peft imports against this transformers
OK   fla-core kernels
OK   FastJEV + OmniJev code


## §2 Download pinned weights (cached on Drive, ~1.9 GB)

In [5]:
import json, os, sys
sys.path.insert(0, "/content/FastJEV")
from huggingface_hub import snapshot_download
cfg = json.load(open("/content/FastJEV/fastjev/configs/0.8B.json"))
CKPT = f"{DRIVE_ROOT}/weights/OmniJev-0.8B"
BASE = f"{DRIVE_ROOT}/weights/Qwen3.5-0.8B"
for art, path in (("adapter", CKPT), ("base", BASE)):
    a = cfg["artifacts"][art]
    snapshot_download(a["repo"], revision=a["revision"], local_dir=path)
    print(art, a["repo"], a["revision"][:10], "->", path)
# FastJEV verifies every file's SHA-256 against the config without importing torch
!cd /content/FastJEV && python -m fastjev --scale 0.8B --checkpoint "{CKPT}" --base-model "{BASE}" --verify-only

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 16 files:   0%|          | 0/16 [00:00<?, ?it/s]

adapter tinnel123/OmniJev-0.8B cb9ac046d6 -> /content/drive/MyDrive/ese6800_omnijev/weights/OmniJev-0.8B


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

base Qwen/Qwen3.5-0.8B 2fc0636471 -> /content/drive/MyDrive/ese6800_omnijev/weights/Qwen3.5-0.8B
{
  "adapter": {
    "repo": "tinnel123/OmniJev-0.8B",
    "revision": "cb9ac046d65469a0a6d2babcfd24c176cd30b0ef",
    "verified_files": 10
  },
  "base": {
    "repo": "Qwen/Qwen3.5-0.8B",
    "revision": "2fc06364715b967f1860aea9cf38778875588b17",
    "verified_files": 10
  }
}


## §3 Smoke test

In [9]:
import os, sys, json, time, math, random
sys.path.insert(0, "/content/FastJEV")
import torch
from fastjev import FastJEV

t0 = time.time()
dense = FastJEV(CKPT, BASE, scale="0.8B", mode="original", protocol="public")
print(f"loaded in {time.time()-t0:.1f}s | branch path: {dense.model.branch} | "
      f"weights on GPU: {torch.cuda.memory_allocated()/2**20:.0f} MiB")

from PIL import Image, ImageDraw
img = Image.new("RGB", (448, 448), "white"); d = ImageDraw.Draw(img)
d.ellipse((120, 120, 330, 330), fill="red")
img.save("/content/smoke.png")
for q in ({"type": "choice", "instructions": "What shape is in the image?",
           "options": [{"key": "A", "text": "a circle"}, {"key": "B", "text": "a square"}, {"key": "C", "text": "a triangle"}]},
          {"type": "noul", "instructions": "There is a red object."},
          {"type": "noul", "instructions": "There is a dog."}):
    r = dense.predict({"images": ["/content/smoke.png"], "question": q}, image_root="/")
    print(q["instructions"], "->", r["answers"]["q"]["prediction"],
          {k: round(v, 3) for k, v in r["answers"]["q"]["probabilities"].items()})
# Expect: circle with high probability, red ~yes, dog ~no.

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

loaded in 20.8s | branch path: True | weights on GPU: 1754 MiB
What shape is in the image? -> A {'A': 1.0, 'B': 0.0, 'C': 0.0, '__abstain__': 0.0}
There is a red object. -> yes {'yes': 0.955, 'no': 0.045}
There is a dog. -> no {'yes': 0.012, 'no': 0.988}


## §4 Reproduce AI2D + POPE (dense)
Datasets come from the `lmms-lab` mirrors on Hugging Face. FastJEV did not release its converters, so our prompt
format is a close guess (question + lettered option list); expect accuracy within ~1–2 points of the paper, not bit-exact.
POPE questions ("Is there a X in the image?") are turned into the statements a `noul` question expects ("There is a X in the image.").

Results are appended per example to a JSONL on Drive, so a disconnected session resumes where it stopped.

> ECE here is on raw head probabilities (FastJEV applies no serving temperature). OmniJev's published ECE (1.33–2.90%)
> is on its own calibration split, which does not appear to be public; treat our AI2D/POPE ECE as the reference we
> compare pruned models against.

In [10]:
import re, io, hashlib
from datasets import load_dataset
IMG_DIR = "/content/imgs"; os.makedirs(IMG_DIR, exist_ok=True)

def _save(img, key):
    p = f"{IMG_DIR}/{key}.png"
    if not os.path.exists(p):
        img.convert("RGB").save(p)
    return p

LETTERS = "ABCDEFGH"

def ai2d_items(n=None):
    ds = load_dataset("lmms-lab/ai2d", split="test")
    print("AI2D columns:", ds.column_names, len(ds))
    idx = list(range(len(ds)))
    if n: idx = random.Random(SEED).sample(idx, n)
    for i in idx:
        r = ds[i]
        opts = list(r["options"])
        gold = int(r["answer"])
        listing = "\n".join(f"{LETTERS[j]}. {o}" for j, o in enumerate(opts))
        yield {"id": f"ai2d-{i}", "gold": LETTERS[gold],
               "request": {"images": [_save(r["image"], f"ai2d-{i}")],
                           "question": {"type": "choice", "instructions": f"{r['question']}\n{listing}",
                                        "options": [{"key": LETTERS[j], "text": o} for j, o in enumerate(opts)]}}}

def pope_statement(q):
    q = q.strip()
    m = re.match(r"(?i)^is there (an?|any|some)?\s*(.+?)\s+in the (image|picture|photo)\?$", q)
    if m:
        art = (m.group(1) or "a").lower()
        return f"There is {art} {m.group(2)} in the {m.group(3).lower()}."
    return q  # fallback: keep the question text

def pope_items(n=None):
    ds = load_dataset("lmms-lab/POPE", split="test")
    print("POPE columns:", ds.column_names, len(ds))
    idx = list(range(len(ds)))
    if n: idx = random.Random(SEED).sample(idx, n)
    for i in idx:
        r = ds[i]
        key = r.get("image_source") or hashlib.md5(r["image"].tobytes()).hexdigest()[:12]
        yield {"id": f"pope-{i}", "gold": r["answer"].strip().lower(), "category": r.get("category"),
               "request": {"images": [_save(r["image"], f"pope-{key}")],
                           "question": {"type": "noul", "instructions": pope_statement(r["question"])}}}

print(pope_statement("Is there a dining table in the image?"))

There is a dining table in the image.


In [11]:
# Metrics: accuracy, ECE (15 equal-width bins on the predicted-class confidence), Brier, NLL
import numpy as np

def option_probs(ans, kind):
    p = dict(ans["probabilities"])
    if kind == "choice":
        abst = p.pop("__abstain__", 0.0)
        s = sum(p.values()) or 1.0
        return {k: v / s for k, v in p.items()}, abst      # renormalised over the supplied options
    return p, 0.0                                           # noul: {"yes", "no"}

def ece(conf, correct, bins=15):
    conf, correct = np.asarray(conf, float), np.asarray(correct, float)
    edges = np.linspace(0, 1, bins + 1); e = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi) if lo > 0 else (conf >= lo) & (conf <= hi)
        if m.any():
            e += m.mean() * abs(conf[m].mean() - correct[m].mean())
    return e

def summarize(rows):
    conf, corr, brier, nll, abst = [], [], [], [], []
    for r in rows:
        p = r["p"]; pred = max(p, key=p.get)
        conf.append(p[pred]); corr.append(pred == r["gold"])
        brier.append(sum((v - (k == r["gold"])) ** 2 for k, v in p.items()))
        nll.append(-math.log(max(p.get(r["gold"], 0.0), 1e-12)))
        abst.append(r.get("abstain", 0.0))
    return {"n": len(rows), "acc": 100 * float(np.mean(corr)), "ece": 100 * ece(conf, corr),
            "brier": float(np.mean(brier)), "nll": float(np.mean(nll)), "mean_abstain": float(np.mean(abst)),
            "mean_conf": float(np.mean(conf))}

In [12]:
def run_eval(engine, tag, items, path):
    done = set()
    if os.path.exists(path):
        done = {json.loads(l)["id"] for l in open(path)}
    kind_t = None
    with open(path, "a") as f:
        for k, it in enumerate(items):
            if it["id"] in done: continue
            torch.cuda.synchronize(); t = time.perf_counter()
            ans = engine.predict(it["request"], image_root="/")["answers"]["q"]
            torch.cuda.synchronize(); dt = time.perf_counter() - t
            p, abst = option_probs(ans, it["request"]["question"]["type"])
            f.write(json.dumps({"id": it["id"], "gold": it["gold"], "p": p, "abstain": abst,
                                "category": it.get("category"), "ms": 1000 * dt}) + "\n")
            if k % 200 == 0: f.flush(); print(tag, k, f"{1000*dt:.0f} ms")
    rows = [json.loads(l) for l in open(path)]
    s = summarize(rows); s["p50_ms"] = float(np.median([r["ms"] for r in rows]))
    return s

SUMMARY = f"{OUT}/summary.json"
summary = json.load(open(SUMMARY)) if os.path.exists(SUMMARY) else {}
PAPER = {"ai2d": 62.27, "pope": 84.76}
for name, gen in (("ai2d", ai2d_items), ("pope", pope_items)):
    s = run_eval(dense, name, gen(N_EVAL), f"{OUT}/dense_{name}.jsonl")
    summary[f"dense/{name}"] = s; json.dump(summary, open(SUMMARY, "w"), indent=1)
    print(f"{name}: acc {s['acc']:.2f}% (FastJEV paper {PAPER[name]}%) | ECE {s['ece']:.2f}% | "
          f"Brier {s['brier']:.4f} | p50 {s['p50_ms']:.0f} ms")
# Go/no-go: full-split accuracy within ~2 points of the paper. If not, check prompt format first, then switch to OmniJev-2B.

README.md:   0%|          | 0.00/700 [00:00<?, ?B/s]

data/test-00000-of-00002.parquet: reconstructing file:   0%|          |  0.00B / 62.3MB            

data/test-00000-of-00002.parquet: downloading bytes:           |  0.00B            

data/test-00001-of-00002.parquet: reconstructing file:   0%|          |  0.00B / 77.2MB            

data/test-00001-of-00002.parquet: downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/3088 [00:00<?, ? examples/s]

... (6 outputs left out)

data/test-00002-of-00003.parquet: reconstructing file:   0%|          |  0.00B / 85.0MB            

data/test-00002-of-00003.parquet: downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/9000 [00:00<?, ? examples/s]

POPE columns: ['id', 'question_id', 'question', 'answer', 'image_source', 'image', 'category'] 9000
pope 0 238 ms
pope 200 225 ms
pope: acc 83.49% (FastJEV paper 84.76%) | ECE 8.33% | Brier 0.2644 | p50 222 ms


## §5 FastJEV pruning baseline (6 skipped candidate layers, `independent` = S+P)
The paper reports S+P as its fastest configuration (1.05–1.26× over original). This is the baseline our method must beat on
quality per unit of speedup. Note: per-example latency here includes image preprocessing, like the paper.

In [13]:
del dense; torch.cuda.empty_cache()
fj = FastJEV(CKPT, BASE, scale="0.8B", mode="independent", pruned_layers=6, protocol="public")
print("pruned layers (zero-based):", fj.layers)
for name, gen in (("ai2d", ai2d_items), ("pope", pope_items)):
    s = run_eval(fj, name, gen(N_EVAL), f"{OUT}/fastjev_sp6_{name}.jsonl")
    summary[f"fastjev_sp6/{name}"] = s; json.dump(summary, open(SUMMARY, "w"), indent=1)
    d = summary[f"dense/{name}"]
    print(f"{name}: acc {s['acc']:.2f}% (dense {d['acc']:.2f}) | ECE {s['ece']:.2f}% (dense {d['ece']:.2f}) | "
          f"speedup {d['p50_ms']/s['p50_ms']:.3f}x")
# Calibration under FastJEV pruning is not reported in their paper: this is our first new number.

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

pruned layers (zero-based): [15, 16, 17, 18, 19, 20]
AI2D columns: ['question', 'options', 'answer', 'image'] 3088
ai2d 0 841 ms
ai2d 200 196 ms
ai2d: acc 61.00% (dense 60.56) | ECE 6.37% (dense 5.04) | speedup 1.202x
POPE columns: ['id', 'question_id', 'question', 'answer', 'image_source', 'image', 'category'] 9000
pope 0 190 ms
pope 200 187 ms
pope: acc 82.00% (dense 83.49) | ECE 6.31% (dense 8.33) | speedup 1.195x


## §6 Where does the time go?
CUDA-event hooks time (a) the vision encoder and (b) every language-model call. On the branch path the **first**
language-model call of a request is the shared prefix; every later call is a batch of candidate branches. Whatever is
left of the request is preprocessing + decision head + Python overhead. Timed on AI2D (4 options) and POPE (1 branch).

In [14]:
del fj; torch.cuda.empty_cache()
dense = FastJEV(CKPT, BASE, scale="0.8B", mode="original", protocol="public")
mods = dict(dense.model.model.backbone.named_modules())
vis_name = min((n for n in mods if n.endswith("visual")), key=len)
lm_name  = min((n for n in mods if n.endswith("language_model")), key=len)
print("vision:", vis_name, "| language model:", lm_name)

log = []
def hook_pair(mod, tag):
    def pre(m, args, kwargs):
        e = torch.cuda.Event(enable_timing=True); e.record(); m._t0 = e
        x = kwargs.get("inputs_embeds", kwargs.get("input_ids"))
        if x is None and args: x = args[0]
        m._shape = tuple(x.shape[:2]) if hasattr(x, "shape") else None
    def post(m, args, kwargs, out):
        e = torch.cuda.Event(enable_timing=True); e.record(); log.append((tag, m._t0, e, m._shape))
    mod.register_forward_pre_hook(pre, with_kwargs=True)
    mod.register_forward_hook(post, with_kwargs=True)
hook_pair(mods[vis_name], "vision"); hook_pair(mods[lm_name], "lm")

def profile(items, reps=40, warm=5):
    out = []
    for k, it in enumerate(items):
        if k >= reps + warm: break
        log.clear(); torch.cuda.synchronize(); t = time.perf_counter()
        dense.predict(it["request"], image_root="/"); torch.cuda.synchronize()
        total = 1000 * (time.perf_counter() - t)
        if k < warm: continue
        vis = sum(a.elapsed_time(b) for tag, a, b, _ in log if tag == "vision")
        lm = [(a.elapsed_time(b), s) for tag, a, b, s in log if tag == "lm"]
        out.append({"total": total, "vision": vis, "prefix": lm[0][0] if lm else 0,
                    "branches": sum(x for x, _ in lm[1:]), "prefix_shape": lm[0][1] if lm else None,
                    "branch_shapes": [s for _, s in lm[1:]]})
    keys = ("total", "vision", "prefix", "branches")
    med = {k: float(np.median([o[k] for o in out])) for k in keys}
    med["other"] = med["total"] - med["vision"] - med["prefix"] - med["branches"]
    return med, out[0]

prof = {}
for name, gen in (("ai2d", ai2d_items), ("pope", pope_items)):
    med, ex = profile(gen(60))
    prof[name] = {"median_ms": med, "example_shapes": {"prefix": ex["prefix_shape"], "branches": ex["branch_shapes"]}}
    share = {k: f"{100*v/med['total']:.0f}%" for k, v in med.items() if k != "total"}
    print(name, f"total {med['total']:.1f} ms", share, "| prefix (B,T):", ex["prefix_shape"],
          "| branch calls:", ex["branch_shapes"])
json.dump(prof, open(f"{OUT}/profile.json", "w"), indent=1)
# NB: if the vision encoder runs inside the first language-model call, 'prefix' includes it; vision is then a sub-part.

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

vision: base_model.model.model.visual | language model: base_model.model.model.language_model
AI2D columns: ['question', 'options', 'answer', 'image'] 3088
ai2d total 231.7 ms {'vision': '6%', 'prefix': '40%', 'branches': '40%', 'other': '15%'} | prefix (B,T): (1, 173) | branch calls: [(4, 7)]
POPE columns: ['id', 'question_id', 'question', 'answer', 'image_source', 'image', 'category'] 9000
pope total 224.1 ms {'vision': '6%', 'prefix': '41%', 'branches': '40%', 'other': '13%'} | prefix (B,T): (1, 321) | branch calls: [(1, 3)]


In [15]:
# Optional deeper look: top CUDA kernels for one AI2D request
from torch.profiler import profile as tprof, ProfilerActivity
it = next(ai2d_items(1))
with tprof(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA]) as p:
    dense.predict(it["request"], image_root="/"); torch.cuda.synchronize()
print(p.key_averages().table(sort_by="cuda_time_total", row_limit=15))

AI2D columns: ['question', 'options', 'answer', 'image'] 3088
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                           aten::linear         1.03%       2.681ms        24.41%      63.765ms      61.668us       0.000us         0.00%      45.895ms      44.385us          1034  
                                           aten::matmul         2.06%       5.389ms        19.60%      51.198ms      52.350us       0.000us      

## §7 2:4 sparsity go / no-go
Times a dense vs a 2:4 semi-structured `linear` at Qwen3.5-0.8B's real projection shapes, at the prefix token count
measured in §6 and at branch-sized token counts. **Rule:** if prefix-sized matmuls gain under ~1.1×, speed claims move to
structured pruning and 2:4 stays a compression-quality study.

In [16]:
import torch.nn.functional as F
from torch.sparse import to_sparse_semi_structured
tc = json.load(open(f"{BASE}/config.json")); tc = tc.get("text_config", tc)
H, I = tc["hidden_size"], tc["intermediate_size"]
shapes = {"q/o-like (H x H)": (H, H), "ffn up/gate (I x H)": (I, H), "ffn down (H x I)": (H, I)}
prefix_T = int(prof["ai2d"]["example_shapes"]["prefix"][1])
tokens = sorted({prefix_T, 512, 64, 16})
print("hidden", H, "intermediate", I, "| token counts", tokens)

def mask24(W):
    w = W.abs().reshape(-1, 4); idx = w.topk(2, dim=1).indices
    m = torch.zeros_like(w, dtype=torch.bool).scatter_(1, idx, True)
    return (W.reshape(-1, 4) * m).reshape(W.shape)

def bench(fn, reps=200):
    for _ in range(20): fn()
    s, e = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
    torch.cuda.synchronize(); s.record()
    for _ in range(reps): fn()
    e.record(); torch.cuda.synchronize(); return s.elapsed_time(e) / reps

res = []
for sname, (o, i) in shapes.items():
    W = mask24(torch.randn(o, i, device="cuda", dtype=torch.bfloat16))
    try:
        Ws = to_sparse_semi_structured(W)
    except Exception as ex:
        print("2:4 unsupported here:", ex); break
    for T in tokens:
        Tp = max(32, -(-T // 32) * 32)       # sparse kernels want aligned row counts
        x = torch.randn(Tp, i, device="cuda", dtype=torch.bfloat16)
        d, sp = bench(lambda: F.linear(x, W)), bench(lambda: F.linear(x, Ws))
        res.append({"shape": sname, "tokens": T, "dense_ms": d, "sparse_ms": sp, "speedup": d / sp})
        print(f"{sname:22s} T={T:5d}  dense {d*1000:7.1f} us  2:4 {sp*1000:7.1f} us  {d/sp:4.2f}x")
json.dump({"gpu": gpu, "rows": res}, open(f"{OUT}/sparse_bench.json", "w"), indent=1)

hidden 1024 intermediate 3584 | token counts [16, 64, 173, 512]


/usr/local/lib/python3.13/dist-packages/torch/sparse/semi_structured.py:115: UserWarning: The PyTorch API of SparseSemiStructuredTensor is in prototype stage and will change in the near future. Please open a Github issue for features requests and see our documentation on the torch.sparse module for further information about the project.
  warnings.warn(


q/o-like (H x H)       T=   16  dense    26.4 us  2:4  1200.6 us  0.02x
q/o-like (H x H)       T=   64  dense    26.0 us  2:4  1204.8 us  0.02x
q/o-like (H x H)       T=  173  dense    24.3 us  2:4  1203.1 us  0.02x
q/o-like (H x H)       T=  512  dense    29.2 us  2:4  1218.6 us  0.02x
ffn up/gate (I x H)    T=   16  dense    26.4 us  2:4  1182.6 us  0.02x
ffn up/gate (I x H)    T=   64  dense    24.7 us  2:4  1184.5 us  0.02x
ffn up/gate (I x H)    T=  173  dense    27.4 us  2:4  1198.6 us  0.02x
ffn up/gate (I x H)    T=  512  dense    48.2 us  2:4  1208.9 us  0.04x
ffn down (H x I)       T=   16  dense    32.1 us  2:4  1194.8 us  0.03x
ffn down (H x I)       T=   64  dense    26.8 us  2:4  1199.2 us  0.02x
ffn down (H x I)       T=  173  dense    27.3 us  2:4  1202.5 us  0.02x
ffn down (H x I)       T=  512  dense    62.0 us  2:4  1185.7 us  0.05x


## §8 LoRA merge equivalence
Pruning acts on dense weights, so the rank-32 adapter must be merged first. `merge_adapter()` folds LoRA into the base
weights in place (the PEFT wrapper stays, so OmniJev's code path is unchanged). Expect max |Δp| around 1e-3 in BF16.

In [17]:
items = list(ai2d_items(50))
before = [dense.predict(it["request"], image_root="/")["answers"]["q"]["probabilities"] for it in items]
dense.model.model.backbone.merge_adapter()
after = [dense.predict(it["request"], image_root="/")["answers"]["q"]["probabilities"] for it in items]
diff = max(abs(a[k] - b[k]) for a, b in zip(before, after) for k in a)
flips = sum(max(a, key=a.get) != max(b, key=b.get) for a, b in zip(before, after))
print(f"max |dp| = {diff:.2e} | decision flips {flips}/{len(items)}")

AI2D columns: ['question', 'options', 'answer', 'image'] 3088
max |dp| = 1.66e-02 | decision flips 0/50


In [18]:
# Week 1 summary for the doc / professor update
print(json.dumps({"env": json.load(open(f"{OUT}/env.json")), "summary": summary, "profile": prof}, indent=1)[:4000])

{
 "env": {
  "gpu": "NVIDIA L4, 23034 MiB, 580.82.07, 8.9",
  "python": "3.13.16",
  "started": "Sat Nov  7 17:09:37 2026"
 },
 "summary": {
  "dense/ai2d": {
   "n": 3088,
   "acc": 60.55699481865285,
   "ece": 5.042482213392182,
   "brier": 0.5183449882509112,
   "nll": 0.9882543520754073,
   "mean_abstain": 0.036252124823745666,
   "mean_conf": 0.6516118641660638,
   "p50_ms": 233.20495150005627
  },
  "dense/pope": {
   "n": 5701,
   "acc": 83.49412383792317,
   "ece": 8.326850299702372,
   "brier": 0.26441509585767203,
   "nll": 0.46065488723349796,
   "mean_abstain": 0.0,
   "mean_conf": 0.9149941312944083,
   "p50_ms": 222.07532199990965
  },
  "fastjev_sp6/ai2d": {
   "n": 300,
   "acc": 61.0,
   "ece": 6.373753722437481,
   "brier": 0.5150844565069287,
   "nll": 0.967547205972852,
   "mean_abstain": 0.03082461436589559,
   "mean_conf": 0.6277458058293991,
   "p50_ms": 193.9525989998856
  },
  "fastjev_sp6/pope": {
   "n": 300,
   "acc": 82.0,
   "ece": 6.305712673813104,
   "

In [19]:
import time
from google.colab import output

while True:
    # Clear the cell output and print a timestamp to simulate activity
    output.clear()
    print("Keep-alive active...")
    time.sleep(60)

Keep-alive active...


KeyboardInterrupt: 